### Alpaca Format Dataset - Three Task Approach

This notebook creates an Alpaca-formatted dataset with three distinct tasks:

1. **Detection**: Determine whether code is vulnerable or benign
   - Instruction: Determine whether the following Java code is vulnerable or benign.
   - Input: {Code Snippet}
   - Output: {"vulnerable" / "benign"}

2. **Localization**: Identify the vulnerable line and explain why
   - Instruction: Analyze the code for {CWE ID}. Identify the vulnerable line and explain why.
   - Input: {Code Snippet}
   - Output: {Vulnerable Line + Explanation}

3. **Repair**: Apply a fix for the vulnerability
   - Instruction: Apply a fix for the {CWE ID} vulnerability in the code snippet.
   - Input: {Code Snippet}
   - Output: {Fixed Code Snippet}

In [ ]:
import pandas as pd
# Read the CSV file with fixes
file_path = '/Users/obiedaananbeh/Documents/PhD Work/VulnFixAI/VulnFixAI_Repo/Trining DataSet/dataSet_withFixes.csv'
df = pd.read_csv(file_path)

print(f"Original dataset size: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

Original dataset size: 19999 rows
Columns: ['CWE ID', 'Project Name', 'Vulnerable File', 'Programming Language', 'Line Number', 'Code Snippet', 'Exact Vulnerable Line', 'Description', 'Status', 'code_fix']


In [15]:
# Analyze the dataset composition
print(f"\n{'='*60}")
print("Dataset Composition Analysis:")
print(f"{'='*60}")

# Check BENIGN vs Vulnerable entries
benign_count = len(df[df['CWE ID'] == 'BENIGN'])
vulnerable_count = len(df[df['CWE ID'] != 'BENIGN'])

print(f"Total entries: {len(df)}")
print(f"  - BENIGN (safe code): {benign_count}")
print(f"  - Vulnerable code: {vulnerable_count}")
print(f"\nCWE ID distribution:")
print(df['CWE ID'].value_counts().head(10))


Dataset Composition Analysis:
Total entries: 19999
  - BENIGN (safe code): 10000
  - Vulnerable code: 9999

CWE ID distribution:
CWE ID
BENIGN     10000
CWE-918     1000
CWE-319     1000
CWE-502     1000
CWE-113     1000
CWE-601     1000
CWE-78      1000
CWE-79      1000
CWE-23      1000
CWE-134     1000
Name: count, dtype: int64


In [16]:
# Task 1: Detection - Determine if code is vulnerable or benign
# This task uses ALL entries (both benign and vulnerable)

df_detection = df[['CWE ID', 'Code Snippet']].copy()
df_detection['instruction'] = "Determine whether the following Java code is vulnerable or benign."
df_detection['input'] = df_detection['Code Snippet']

# Correctly label based on CWE ID
df_detection['output'] = df_detection['CWE ID'].apply(
    lambda x: 'benign' if x == 'BENIGN' else 'vulnerable'
)

print(f"Detection task created: {len(df_detection)} examples")
print(f"  - Benign examples: {len(df_detection[df_detection['output'] == 'benign'])}")
print(f"  - Vulnerable examples: {len(df_detection[df_detection['output'] == 'vulnerable'])}")

print("\nSample BENIGN Detection example:")
benign_sample = df_detection[df_detection['output'] == 'benign'].iloc[0]
print(f"Instruction: {benign_sample['instruction']}")
print(f"Input: {benign_sample['input'][:100]}...")
print(f"Output: {benign_sample['output']}")

print("\nSample VULNERABLE Detection example:")
vuln_sample = df_detection[df_detection['output'] == 'vulnerable'].iloc[0]
print(f"Instruction: {vuln_sample['instruction']}")
print(f"Input: {vuln_sample['input'][:100]}...")
print(f"Output: {vuln_sample['output']}")

Detection task created: 19999 examples
  - Benign examples: 10000
  - Vulnerable examples: 9999

Sample BENIGN Detection example:
Instruction: Determine whether the following Java code is vulnerable or benign.
Input: private static final List<String> IGNORED_LOG_FIELDS = Arrays.asList("error message", "realm", "user...
Output: benign

Sample VULNERABLE Detection example:
Instruction: Determine whether the following Java code is vulnerable or benign.
Input: protected void doRetrieveMatchingFiles(String fullPattern, File dir, Set<File> result) throws IOExce...
Output: vulnerable


In [17]:
# Task 2: Localization - Identify the vulnerable line and explain why
# This task uses ONLY vulnerable entries (exclude BENIGN)

df_localization = df[df['CWE ID'] != 'BENIGN'][['CWE ID', 'Code Snippet', 'Exact Vulnerable Line', 'Description']].copy()
df_localization['instruction'] = df_localization.apply(
    lambda row: f"Analyze the code for {row['CWE ID']}. Identify the vulnerable line and explain why.", 
    axis=1
)
df_localization['input'] = df_localization['Code Snippet']
df_localization['output'] = df_localization.apply(
    lambda row: f"{row['Exact Vulnerable Line']}\nExplanation: {row['Description']}", 
    axis=1
)

print(f"Localization task created: {len(df_localization)} examples")
print(f"  - All examples are from vulnerable code (BENIGN entries excluded)")

print("\nSample Localization example:")
print(f"CWE ID: {df_localization.iloc[0]['CWE ID']}")
print(f"Instruction: {df_localization.iloc[0]['instruction']}")
print(f"Input: {df_localization.iloc[0]['input'][:100]}...")
print(f"Output: {df_localization.iloc[0]['output'][:150]}...")

Localization task created: 9999 examples
  - All examples are from vulnerable code (BENIGN entries excluded)

Sample Localization example:
CWE ID: CWE-918
Instruction: Analyze the code for CWE-918. Identify the vulnerable line and explain why.
Input: protected void doRetrieveMatchingFiles(String fullPattern, File dir, Set<File> result) throws IOExce...
Output: doRetrieveMatchingFiles(fullPattern, content, result);
Explanation: Unsanitized input from a zip file flows into openConnection, where it is used as a...


In [18]:
# Task 3: Repair - Apply a fix for the vulnerability
# This task uses ONLY vulnerable entries (exclude BENIGN)

df_repair = df[df['CWE ID'] != 'BENIGN'][['CWE ID', 'Code Snippet', 'code_fix']].copy()
df_repair['instruction'] = df_repair.apply(
    lambda row: f"Apply a fix for the {row['CWE ID']} vulnerability in the code snippet.", 
    axis=1
)
df_repair['input'] = df_repair['Code Snippet']
df_repair['output'] = df_repair['code_fix']

print(f"Repair task created: {len(df_repair)} examples")
print(f"  - All examples are from vulnerable code (BENIGN entries excluded)")

print("\nSample Repair example:")
print(f"CWE ID: {df_repair.iloc[0]['CWE ID']}")
print(f"Instruction: {df_repair.iloc[0]['instruction']}")
print(f"Input: {df_repair.iloc[0]['input'][:100]}...")
print(f"Output: {df_repair.iloc[0]['output'][:150]}...")

Repair task created: 9999 examples
  - All examples are from vulnerable code (BENIGN entries excluded)

Sample Repair example:
CWE ID: CWE-918
Instruction: Apply a fix for the CWE-918 vulnerability in the code snippet.
Input: protected void doRetrieveMatchingFiles(String fullPattern, File dir, Set<File> result) throws IOExce...
Output: import javax.net.ssl.*;
    import java.security.*;
    
    // Initialize security components
    private static final TrustManager[] trustStore = cr...


In [19]:
# Combine all three tasks into a single Alpaca-formatted dataset
# Keep only the required columns: instruction, input, output
df_detection_final = df_detection[['instruction', 'input', 'output']].copy()
df_localization_final = df_localization[['instruction', 'input', 'output']].copy()
df_repair_final = df_repair[['instruction', 'input', 'output']].copy()

# Concatenate all three datasets
df_alpaca = pd.concat([df_detection_final, df_localization_final, df_repair_final], ignore_index=True)

print(f"\n{'='*60}")
print(f"Final Alpaca Dataset Summary:")
print(f"{'='*60}")
print(f"Total examples: {len(df_alpaca)}")
print(f"  - Detection examples: {len(df_detection_final)}")
print(f"    • Benign: {len(df_detection[df_detection['output'] == 'benign'])}")
print(f"    • Vulnerable: {len(df_detection[df_detection['output'] == 'vulnerable'])}")
print(f"  - Localization examples: {len(df_localization_final)} (vulnerable only)")
print(f"  - Repair examples: {len(df_repair_final)} (vulnerable only)")
print(f"\nDataset columns: {df_alpaca.columns.tolist()}")


Final Alpaca Dataset Summary:
Total examples: 39997
  - Detection examples: 19999
    • Benign: 10000
    • Vulnerable: 9999
  - Localization examples: 9999 (vulnerable only)
  - Repair examples: 9999 (vulnerable only)

Dataset columns: ['instruction', 'input', 'output']


In [20]:
# Save the Alpaca-formatted dataset to a CSV file
output_file_path = 'ITV_alpaca_dataset.csv'
df_alpaca.to_csv(output_file_path, index=False)

print(f"\n✅ Alpaca dataset saved to: {output_file_path}")
print(f"\nDataset saved successfully with {len(df_alpaca)} total examples!")

# Display a few examples from each task
print(f"\n{'='*60}")
print("Sample Examples from Each Task:")
print(f"{'='*60}")

print("\n📋 DETECTION Example:")
print(f"Instruction: {df_alpaca.iloc[0]['instruction']}")
print(f"Input (truncated): {df_alpaca.iloc[0]['input'][:80]}...")
print(f"Output: {df_alpaca.iloc[0]['output']}")

print("\n🔍 LOCALIZATION Example:")
idx_loc = len(df_detection_final)
print(f"Instruction: {df_alpaca.iloc[idx_loc]['instruction']}")
print(f"Input (truncated): {df_alpaca.iloc[idx_loc]['input'][:80]}...")
print(f"Output (truncated): {df_alpaca.iloc[idx_loc]['output'][:120]}...")

print("\n🔧 REPAIR Example:")
idx_rep = len(df_detection_final) + len(df_localization_final)
print(f"Instruction: {df_alpaca.iloc[idx_rep]['instruction']}")
print(f"Input (truncated): {df_alpaca.iloc[idx_rep]['input'][:80]}...")
print(f"Output (truncated): {df_alpaca.iloc[idx_rep]['output'][:120]}...")


✅ Alpaca dataset saved to: ITV_alpaca_dataset.csv

Dataset saved successfully with 39997 total examples!

Sample Examples from Each Task:

📋 DETECTION Example:
Instruction: Determine whether the following Java code is vulnerable or benign.
Input (truncated): private static final List<String> IGNORED_LOG_FIELDS = Arrays.asList("error mess...
Output: benign

🔍 LOCALIZATION Example:
Instruction: Analyze the code for CWE-918. Identify the vulnerable line and explain why.
Input (truncated): protected void doRetrieveMatchingFiles(String fullPattern, File dir, Set<File> r...
Output (truncated): doRetrieveMatchingFiles(fullPattern, content, result);
Explanation: Unsanitized input from a zip file flows into openCon...

🔧 REPAIR Example:
Instruction: Apply a fix for the CWE-918 vulnerability in the code snippet.
Input (truncated): protected void doRetrieveMatchingFiles(String fullPattern, File dir, Set<File> r...
Output (truncated): import javax.net.ssl.*;
    import java.security.*;
    
    /

### Optional: Create JSON Format for Alpaca

Some frameworks prefer JSON format over CSV. Here's how to convert the dataset to JSON format compatible with Alpaca fine-tuning.

In [21]:
# Save as JSON format (commonly used for fine-tuning)
import json

json_output_path = 'ITV_alpaca_dataset.json'

# Convert DataFrame to list of dictionaries
alpaca_json = df_alpaca.to_dict('records')

# Save to JSON file with proper formatting
with open(json_output_path, 'w', encoding='utf-8') as f:
    json.dump(alpaca_json, f, indent=2, ensure_ascii=False)

print(f"✅ JSON dataset saved to: {json_output_path}")
print(f"Total examples in JSON: {len(alpaca_json)}")

✅ JSON dataset saved to: ITV_alpaca_dataset.json
Total examples in JSON: 39997
